In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from renormalizer.utils import constant

NOTEBOOK_DIR = Path.cwd().resolve()
DATA_DIR = NOTEBOOK_DIR.parent / "emi" / "singleset"
files = sorted(p for p in DATA_DIR.glob("Dimer_emi_s*_offset_*.npz") if "_impo" not in p.name and "thermal" not in p.name)
if not files:
    raise FileNotFoundError(f"No offset spectra found in {DATA_DIR}")

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "pdf.fonttype": 42, "ps.fonttype": 42})
print("Found files:")
for p in files:
    print(" ", p.relative_to(DATA_DIR.parent.parent))


In [ ]:
def scalar(data, key):
    return np.asarray(data[key]).item()

def load_time(data):
    return np.asarray(data["time_series" if "time_series" in data.files else "time series"], dtype=float)

def phase_value(data, phase_mode):
    eex0 = float(scalar(data, "e_ex0_energy"))
    e00 = float(scalar(data, "e00_energy"))
    zpe = float(scalar(data, "gs_zpe_energy"))
    carrier = float(scalar(data, "carrier_energy"))
    return {
        "E_EX0": eex0,
        "E00": e00,
        "E00+ZPE": e00 + zpe,
        "ZPE": zpe,
        "carrier": carrier,
        "none": 0.0,
    }[phase_mode]

def spectrum(path, phase_mode, freq_sign, undo_saved_conj):
    data = np.load(path, allow_pickle=True)
    t = load_time(data)
    tau = t - t[0]
    dt = t[1] - t[0]
    corr = np.asarray(data["autocorr"], dtype=np.complex128)
    if undo_saved_conj:
        corr = np.conjugate(corr)
    corr = corr * np.exp(1j * phase_value(data, phase_mode) * tau)
    fft_freq = np.fft.fftfreq(len(corr), d=dt) * 2.0 * np.pi
    freq = freq_sign * fft_freq
    y = np.abs(np.fft.fft(corr))
    keep = freq > 0
    x = 1e7 / (freq[keep] * constant.au2cm)
    y = y[keep] * freq[keep] ** 3
    y = y / y.max()
    order = np.argsort(x)
    return x[order], y[order]

def local_peaks(x, y, window=(250, 1200), n=6):
    mask = (x >= window[0]) & (x <= window[1])
    xs, ys = x[mask], y[mask]
    idx = [i for i in range(1, len(ys) - 1) if ys[i] >= ys[i-1] and ys[i] >= ys[i+1]]
    idx = sorted(idx, key=lambda i: ys[i], reverse=True)[:n]
    return [(float(xs[i]), float(ys[i])) for i in idx]

def file_label(path):
    data = np.load(path, allow_pickle=True)
    return f"chi={int(scalar(data, 'max_bonddim'))}, offset={scalar(data, 'offset_mode')}"


In [ ]:
tests = [
    {"name": "old sign, undo-conj, +E_EX0", "phase": "E_EX0", "freq_sign": -1, "undo": True, "color": "#0570b0", "ls": "-"},
    {"name": "old sign, saved direct, +E_EX0", "phase": "E_EX0", "freq_sign": -1, "undo": False, "color": "#74a9cf", "ls": "--"},
    {"name": "opposite sign, undo-conj, +E00", "phase": "E00", "freq_sign": 1, "undo": True, "color": "#e6550d", "ls": "-"},
    {"name": "opposite sign, saved direct, +E00", "phase": "E00", "freq_sign": 1, "undo": False, "color": "#fdae6b", "ls": "--"},
    {"name": "opposite sign, undo-conj, +E_EX0", "phase": "E_EX0", "freq_sign": 1, "undo": True, "color": "#31a354", "ls": ":"},
]

for path in files:
    data = np.load(path, allow_pickle=True)
    print("\n" + file_label(path))
    print(f"  carrier = {float(scalar(data, 'carrier_energy')) * constant.au2ev:.9f} eV")
    print(f"  E_EX0   = {float(scalar(data, 'e_ex0_energy')) * constant.au2ev:.9f} eV")
    print(f"  E00     = {float(scalar(data, 'e00_energy')) * constant.au2ev:.9f} eV")
    print(f"  ZPE     = {float(scalar(data, 'gs_zpe_energy')) * constant.au2ev:.9f} eV")
    for test in tests:
        x, y = spectrum(path, test["phase"], test["freq_sign"], test["undo"])
        print(f"  {test['name']:<36}", [f"{px:.1f}:{py:.3f}" for px, py in local_peaks(x, y)])


In [ ]:
fig, axes = plt.subplots(len(files), 1, figsize=(8.2, 3.8 * len(files)), squeeze=False, constrained_layout=True)
for ax, path in zip(axes[:, 0], files):
    for test in tests:
        x, y = spectrum(path, test["phase"], test["freq_sign"], test["undo"])
        ax.plot(x, y, label=test["name"], color=test["color"], linestyle=test["ls"], linewidth=1.8)
    ax.set_xlim(250, 750)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Normalized intensity")
    ax.set_title(file_label(path))
    ax.grid(alpha=0.22, linewidth=0.7)
    ax.legend(frameon=False, fontsize=8)
fig.savefig("dimer_emi_offset_phase_compare.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# Final aligned emission plot: use the saved finite-T emission autocorr directly,
# positive FFT frequency axis, and add back E00 for the E_EX0-offset run.
fig, ax = plt.subplots(figsize=(7.0, 4.4), constrained_layout=True)
for path in files:
    x, y = spectrum(path, "E00", 1, False)
    peaks = local_peaks(x, y, window=(520, 700), n=4)
    peak_text = peaks[0][0] if peaks else float('nan')
    ax.plot(x, y, label=f"{file_label(path)}, peak={peak_text:.1f} nm", linewidth=2.0)
    print(f"Aligned final {file_label(path)}", [f"{px:.1f}:{py:.3f}" for px, py in peaks])
ax.set_xlim(520, 700)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Normalized intensity")
ax.set_title("PBI dimer offset emission aligned to Ren 2018 Fig.4")
ax.grid(alpha=0.22, linewidth=0.7)
ax.legend(frameon=False, fontsize=9)
fig.savefig("dimer_emi_offset_aligned_fig4.pdf", bbox_inches="tight")
plt.show()
